# QuantShield Exploratory Analysis

Inspect local prices and returns, then add technical-analysis and simple ML dashboards for a chosen asset.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from quantshield.config import load_config
from quantshield.notebook_analysis import (
    build_ta_feature_frame,
    fit_ml_return_model,
    plot_ml_dashboard,
    plot_price_technical_dashboard,
)
from quantshield.pipeline import prepare_market_data

config = load_config(ROOT / 'config' / 'default_config.yaml')
prices, returns = prepare_market_data(config)
prices.tail(), returns.tail()

(                   VOO         IVV         SPY         VTI         QQQ  \
 Date                                                                     
 2026-04-06  605.669983  661.859985  658.929993  325.209991  588.500000   
 2026-04-07  606.039978  662.380005  659.219971  325.429993  588.590027   
 2026-04-08  621.340027  678.989990  676.010010  333.700012  606.090027   
 2026-04-09  625.020020  683.010010  679.909973  335.450012  610.190002   
 2026-04-10  624.599976  682.619995  679.460022  335.049988  611.070007   
 
                   VEA         VUG         GLD       IEFA         VTV  
 Date                                                                  
 2026-04-06  65.120003  444.079987  427.649994  91.889999  197.839996  
 2026-04-07  65.089996  444.929993  431.809998  91.750000  197.580002  
 2026-04-08  67.820000  456.899994  434.529999  95.419998  201.979996  
 2026-04-09  67.550003  459.519989  437.910004  95.139999  202.949997  
 2026-04-10  67.739998  461.130005  437.1

In [2]:
returns.corr()

,VOO,IVV,SPY,VTI,QQQ,VEA,VUG,GLD,IEFA,VTV
VOO,1.000000,0.999467,0.998844,0.995644,0.932966,0.855340,0.955167,0.052104,0.844187,0.925680
IVV,0.999467,1.000000,0.998937,0.995684,0.933108,0.855689,0.955134,0.053249,0.844712,0.925413
SPY,0.998844,0.998937,1.000000,0.996088,0.934299,0.855045,0.956269,0.052524,0.844367,0.922787
VTI,0.995644,0.995684,0.996088,1.000000,0.929037,0.860433,0.953868,0.056440,0.849261,0.923290
QQQ,0.932966,0.933108,0.934299,0.929037,1.000000,0.762700,0.981175,0.062146,0.753043,0.750540
VEA,0.855340,0.855689,0.855045,0.860433,0.762700,1.000000,0.786527,0.172855,0.995813,0.839068
VUG,0.955167,0.955134,0.956269,0.953868,0.981175,0.786527,1.000000,0.062352,0.776979,0.779032
GLD,0.052104,0.053249,0.052524,0.056440,0.062146,0.172855,0.062352,1.000000,0.158101,0.035987
IEFA,0.844187,0.844712,0.844367,0.849261,0.753043,0.995813,0.776979,0.158101,1.000000,0.827931
VTV,0.925680,0.925413,0.922787,0.923290,0.750540,0.839068,0.779032,0.035987,0.827931,1.000000


## Price, Technical Analysis, and ML View

In [3]:
ticker = config.backtest.benchmark_ticker if config.backtest.benchmark_ticker in prices.columns else prices.columns[0]
analysis_frame = build_ta_feature_frame(prices[ticker], returns[ticker])
plot_price_technical_dashboard(analysis_frame, ticker)
analysis_frame[[
    'price',
    'sma_20',
    'sma_50',
    'rsi_14',
    'macd',
    'macd_signal',
    'macd_hist',
]].tail(10)


,price,sma_20,sma_50,rsi_14,macd,macd_signal,macd_hist
Date,,,,,,,
2026-03-26,645.090027,667.209869,678.973721,33.168151,-8.501991,-6.777369,-1.724622
2026-03-27,634.090027,664.708289,677.885927,28.529763,-9.855153,-7.392926,-2.462227
2026-03-30,631.969971,662.081256,676.718234,27.725020,-10.972134,-8.108768,-2.863367
2026-03-31,650.340027,660.674402,675.929510,42.784787,-10.256806,-8.538375,-1.718431
2026-04-01,655.239990,659.273203,675.519619,46.015978,-9.188596,-8.668419,-0.520177
2026-04-02,655.830017,658.091983,674.965554,46.408452,-8.199899,-8.574715,0.374816
2026-04-06,658.929993,657.511047,674.402084,48.525867,-7.084541,-8.276680,1.192139
2026-04-07,659.219971,656.650909,673.839427,48.729930,-6.106817,-7.842708,1.735890
2026-04-08,676.010010,656.684628,673.542762,58.891840,-3.931825,-7.060531,3.128706


In [4]:
model, evaluation, metrics, feature_importance = fit_ml_return_model(analysis_frame)
plot_ml_dashboard(evaluation, feature_importance, ticker)
pd.concat([metrics, feature_importance.rename_axis('feature').to_frame().T], axis=0)


,train_samples,test_samples,mean_absolute_error,r_squared,buy_and_hold_total_return,ml_strategy_total_return,prediction_hit_rate,sma_50_gap,return_lag_5,sma_20_gap,bb_width,rsi_14,macd,volatility_20,macd_hist,return_lag_10,return_lag_1
metrics,2226.0,557.0,0.006647,0.003181,0.447171,0.369842,0.545781,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ridge_coefficient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.013077,-0.012921,-0.012654,-0.001549,-0.00002,0.00003,0.000266,0.000298,0.014574,0.017701
